# Gridworld


In Reinforcement Learning (RL), a GridWorld is the "Hello World" of environments. It is a discrete, two-dimensional space where an agent moves from cell to cell to maximize its rewards.

It serves as the perfect bridge between theory and practice because every component of a Markov Decision Process (MDP) is visible and countable.

A GridWorld is defined by four core mathematical components:

- States (S): Each cell in the grid (e.g., (x,y) or a flattened index from 0 to 15).

- Actions (A): Usually four discrete moves: Up, Down, Left, and Right.

- Transitions (P): The rules of movement. In a simple grid, this is deterministic (moving 'Right' always lands you in the cell to the right). If you hit a wall, you stay put.

- Rewards (R): The feedback signal.

- Step Penalty: Usually −1 per move to force the agent to find the shortest path.

- Goal: A large positive reward or 0 (with penalties elsewhere).

- Traps: Large negative rewards (e.g., −10) that may end the episode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns

# clean 255
# goal 200


def generate_enclosed_map(rows=30, cols=30, wall_density=0.25, trap_density=0.02):
    # Start with a clean white path (255)
    grid = np.full((rows, cols), 255, dtype=np.uint8)
    
    grid[0, :] = 0   # Top wall
    grid[-1, :] = 0  # Bottom wall
    grid[:, 0] = 0   # Left wall
    grid[:, -1] = 0  # Right wall
    
    interior_shape = (rows-2, cols-2)
    random_roll = np.random.rand(*interior_shape)
    
    interior_mask = np.full((rows, cols), False)
    interior_mask[1:-1, 1:-1] = True
    
    grid[(interior_mask) & (np.pad(random_roll < wall_density, 1, constant_values=False))] = 0
    grid[(interior_mask) & (np.pad(random_roll > 1 - trap_density, 1, constant_values=False))] = 50
    
    grid[-2, -2] = 200    # Bottom-right goal
    
    return grid

rows, cols = 10, 10
grid_bitmap = generate_enclosed_map(rows, cols)
plt.imshow(grid_bitmap, cmap='gray')
plt.title("Enclosed GridWorld with Random Obstacles")

plt.figure(figsize=(9, 9))
plt.imshow(grid_bitmap, cmap='gray', vmin=0, vmax=255)

rows, cols = grid_bitmap.shape

for r in range(rows):
    for c in range(cols):
        state = r * cols + c

        plt.text(
            c, r, str(state),
            ha='center',
            va='center',
            color='red',
            fontsize=8
        )

plt.title("GridWorld - State Numbering")
plt.xticks(range(cols))
plt.yticks(range(rows))


plt.show()

In [ ]:
# define rewards
WORLD_CONFIG = {
    0:   {"reward": 0,      "terminal": False, "label": "Wall"},   # Penalty handled by 'stay-put'
    50:  {"reward": -10,    "terminal": True,  "label": "Trap"},   # Big penalty, ends game
    200: {"reward": 10,     "terminal": True,  "label": "Goal"},   # Big prize, ends game
    255: {"reward": -1,     "terminal": False, "label": "Path"}    # Step cost to encourage speed
}

# convert config to two maps like the one we generated 
def parse_grid(grid_bitmap):
    rows, cols = grid_bitmap.shape
    reward_grid = np.zeros((rows, cols))
    terminal_mask = np.zeros((rows, cols), dtype=bool)

    for val, props in WORLD_CONFIG.items():
        # Masking: find all coordinates where the bitmap matches the value
        mask = (grid_bitmap == val)
        reward_grid[mask] = props["reward"]
        terminal_mask[mask] = props["terminal"]
        
    return reward_grid, terminal_mask

reward_grid, terminal_mask = parse_grid(grid_bitmap)

In [ ]:
MOVEMENTS = [(-1, 0), (1, 0), (0, -1), (0,1)]
#               UP -- DOWN -- LEFT -- RIGHT  

def step(s, action_idx, grid_bitmap, reward_grid, terminal_mask):
    #conert the index s to a row, col position and retrieve rewards

    
    return s_next, reward, done

def simulate_episode(start_state, grid_bitmap, reward_grid, terminal_mask, policy, gamma=0.9, max_steps=100):
    state = start_state
    episode_history = []
    
    for t in range(max_steps):
        
    # Calculate Return G_t = sum of discounted rewards
    G = 0
    # We calculate backwards for efficiency: G = r + gamma * G

    # Return starting state and the total discounted return from t=0
    return returns[-1], transitions

In [ ]:
def random_policy(s):  # Random policy for testing
    return np.random.choice(len(MOVEMENTS))

G_final, episode_history = simulate_episode(12, grid_bitmap, reward_grid, terminal_mask, random_policy, gamma=0.9, max_steps=100)

In [ ]:
episode_history

## Solving the problem with the bellman equations

This is the "mathematical engine" of the lab. Since we are using a Fixed Random Policy, the Bellman Equation transitions from a complex optimization problem to a straightforward system of linear equations.1. The Bellman Expectation EquationFor a fixed policy $\pi$, the value of a state $s$ is the expected sum of future discounted rewards. Mathematically, it is expressed as:$$V^{\pi}(s) = \sum_{a \in A} \pi(a|s) \sum_{s' \in S, r \in R} p(s', r|s, a) \left[ r + \gamma V^{\pi}(s') \right]$$


We are using a Uniform Random Policy ($\pi(a|s) = 0.25$) and the environment is Deterministic ($p=1$ for the result of an action), the equation simplifies to:$$V(s) = 0.25 \sum_{a \in \{U,D,L,R\}} \left( \text{Reward}(s, a) + \gamma V(s_{next}) \right)$$ 


To solve this for all $N$ states simultaneously, we rewrite it in matrix-vector form. This turns $16$ (or $100$) individual equations into one single operation:$$\mathbf{V} = \mathbf{R} + \gamma \mathbf{P} \mathbf{V}$$


Where:

- $\mathbf{V}$: An $N \times 1$ vector of values we want to find.

- $\mathbf{R}$: An $N \times 1$ vector of expected rewards for each state under the random policy.

- $\mathbf{P}$: An $N \times N$ Transition Probability Matrix, where $P_{ij}$ is the probability of landing in state $j$ starting from state $i$.

- $\gamma$: The discount factor (e.g., $0.9$). 

P encodes our policy, in this case it is a matrix which has entries 0 for each element unreachable and the probability given by the policy for the others! Second note we can factor out an "expected reward" since we can loop on all the states from there and factor out this R vector (which results from the multipltication $PR^*$ where $R^*$ contains all the rewards)

To find $\mathbf{V}$, we rearrange the equation using linear algebra:Move the $V$ terms to one side: $$ \mathbf{V} - \gamma \mathbf{P} \mathbf{V} = \mathbf{R}$$ Factor out $\mathbf{V}$: $$(\mathbf{I} - \gamma \mathbf{P}) \mathbf{V} = \mathbf{R}$$ Solve for $\mathbf{V}$: $$\mathbf{V} = (\mathbf{I} - \gamma \mathbf{P})^{-1} \mathbf{R} $$ 



In [ ]:
def build_matrices_from_random_policy(grid_bitmap, reward_grid, terminal_mask, gamma=0.9):
    rows, cols = grid_bitmap.shape
    num_states = rows * cols

    # Initialize P as an N x N matrix and R as an N x 1 vector
    P = np.zeros((num_states, num_states))
    R_vec = np.zeros(num_states)

    for s in range(num_states):
        r, c = divmod(s, cols)
        
        # In an MDP, terminal states transition to themselves with prob 1.0
        if terminal_mask[r, c] or grid_bitmap[r, c] == 0: # it is a wall or terminal state
            P[s, s] = 1.0
            continue
            
        prob_per_action = ...
        for a_idx in range(4):
            # Use our step logic to find where this action leads
            # dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][a_idx]
            
            
            # Update Transition Matrix: P[from_state, to_state]

            
            # Update Reward Vector: R[state] is the expected immediate reward
            # R(s) = sum_{a} pi(a|s) * Reward(s, a)

    return P, R_vec


def solve_bellman_analytical(P, R_vec, gamma=0.9):
    """
    Solves (I - gamma*P)V = R
    """
    num_states = P.shape[0]
    I = np.eye(num_states)
    
    # Solve the linear system Ax = b 
    # where A = (I - gamma*P) and b = R_vec
    
    return V_exact

In [ ]:
P, R_vec = build_matrices_from_random_policy(grid_bitmap, reward_grid, terminal_mask)
V_exact = solve_bellman_analytical(P, R_vec, gamma=0.9)

V_exact_grid = V_exact.reshape(rows, cols)

R_expected_grid = R_vec.reshape(rows, cols)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Define the wall mask
wall_mask = (grid_bitmap == 0)

# --- Plot 1: Expected Rewards (R_vec) ---
sns.heatmap(R_expected_grid, mask=wall_mask, annot=True, fmt=".1f", 
            cmap="RdYlGn", ax=ax[0], cbar_kws={'label': 'Expected Immediate Reward'})
ax[0].set_title(r"Expected Reward Map ($R$ vector)")

# --- Plot 2: Exact Value Function (V) ---
sns.heatmap(V_exact_grid, mask=wall_mask, annot=True, fmt=".1f",
            cmap="viridis", ax=ax[1], cbar_kws={'label': 'Discounted Future Return (V)'})
ax[1].set_title(r"Analytical Value Function ($V$) Solved via $(I - \gamma P)^{-1} R$")

plt.show()